# Lab 01: Document Parsing & Chunking

## Overview
Parse arXiv AI/ML papers stored in a Unity Catalog Volume into structured data, then chunk the content for downstream vector search indexing.

## Learning Objectives
- Use `ai_parse_document()` to extract text, tables, and images from PDFs
- Compare SQL and Python parsing approaches
- Clean parsed content using `ai_query()` for semantic cleaning
- Implement recursive character chunking with configurable overlap
- Save flattened chunks to a Delta table ready for embedding

## Exam Domain
**Data Preparation (14%)** — This lab covers document ingestion, parsing, transformation, and chunking — core skills tested in the Data Preparation section.

In [ ]:
# Prerequisites — run once per session
%pip install databricks-sdk mlflow langchain langchain-text-splitters --quiet
dbutils.library.restartPython()

> **Cost tip:** These labs run on **serverless compute** by default — no cluster setup needed. You only pay per-second of actual usage. The `COST_PROFILE` below lets you choose cheaper models if you're cost-sensitive.

In [ ]:
# === Cost Profile ===
# "budget"   — cheapest models, may have lower quality (~$0.50/lab)
# "balanced" — good quality/cost ratio (~$1-2/lab)  [DEFAULT]
# "quality"  — best models, highest cost (~$3-5/lab)
COST_PROFILE = "balanced"

_LLM_ENDPOINTS = {
    "budget":   "databricks-meta-llama-3-1-8b-instruct",
    "balanced": "databricks-meta-llama-3-1-8b-instruct",
    "quality":  "databricks-meta-llama-3-3-70b-instruct",
}
LLM_ENDPOINT = _LLM_ENDPOINTS[COST_PROFILE]

# Configuration — shared across all labs
CATALOG = "genai_lab_guide"
SCHEMA = "default"
VOLUME = "arxiv_papers"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Verify the volume exists and list papers
display(spark.sql(f"LIST '{VOLUME_PATH}'"))

## A. Parsing Documents with Python

We use `ai_parse_document()` to extract structured content from PDFs. This Databricks SQL function leverages Mosaic AI to detect text, tables, and images automatically.

The Python approach uses `expr()` to call the SQL function from a Spark DataFrame.

In [ ]:
from pyspark.sql.functions import expr

# Read all PDFs as binary files
docs_df = spark.read.format("binaryFile").load(VOLUME_PATH)

print(f"Found {docs_df.count()} documents")
display(docs_df.select("path", "length"))

In [ ]:
from pyspark.sql.functions import expr

# Read all PDFs as binary files
docs_df = spark.read.format("binaryFile").load(VOLUME_PATH)
print(f"Found {docs_df.count()} documents")

# Parse each document and save to table (VARIANT type needs SQL for best compat)
parsed_df = docs_df.withColumn(
    "parsed_content",
    expr("ai_parse_document(content, map('version', '2.0'))")
)

PARSED_TABLE = f"{CATALOG}.{SCHEMA}.parsed_docs"
parsed_df.select("path", "parsed_content").write.mode("overwrite").saveAsTable(PARSED_TABLE)
print(f"Parsed documents saved to {PARSED_TABLE}")

# Show parsing results using SQL (handles VARIANT natively)
display(spark.sql(f"""
SELECT
  path,
  parsed_content:error_status::STRING AS error_status
FROM {PARSED_TABLE}
"""))

## B. Parsing with SQL

The same operation in pure SQL — ideal for batch processing and scheduled jobs.

In [ ]:
# Alternative: pure SQL approach (same result, different syntax)
display(spark.sql(f"""
SELECT
  path,
  ai_parse_document(content, map('version', '2.0')):error_status::STRING AS error_status
FROM read_files('{VOLUME_PATH}', format => 'binaryFile')
"""))

## C. Inspecting Parsed Output

The `ai_parse_document` output contains rich metadata:
- **`document:pages`** — list of page objects with `page_number`, `text`, `tables`
- **`document:elements`** — individual elements (paragraphs, headings, tables, images)
- **`error_status`** — any parsing errors
- **`metadata`** — document-level metadata

In [ ]:
# Extract key metadata fields using SQL (VARIANT-safe)
display(spark.sql(f"""
SELECT
  path,
  parsed_content:error_status::STRING AS error_status,
  parsed_content:metadata::STRING AS metadata
FROM {CATALOG}.{SCHEMA}.parsed_docs
"""))

In [ ]:
# Extract text from all pages using SQL LATERAL VIEW + VARIANT
pages_df = spark.sql(f"""
SELECT
  path,
  page.page_number::INT AS page_number,
  page.text::STRING AS page_text
FROM {CATALOG}.{SCHEMA}.parsed_docs
LATERAL VIEW explode(from_json(
  parsed_content:document:pages::STRING,
  'ARRAY<STRUCT<page_number: INT, text: STRING>>'
)) AS page
""")

pages_df.createOrReplaceTempView("pages")
display(pages_df)

## D. Semantic Cleaning with ai_query()

Raw parsed text often contains artifacts: broken lines, orphaned headers, inconsistent spacing. Using `ai_query()` with a Foundation Model, we can clean the text semantically — preserving structure while removing noise.

This is higher quality than simple regex/concatenation but costs more (LLM tokens).

In [ ]:
from pyspark.sql.functions import concat_ws, collect_list

# Concatenate all page text per document
raw_text_df = pages_df.groupBy("path").agg(
    concat_ws("\n\n", collect_list("page_text")).alias("raw_text")
)
raw_text_df.createOrReplaceTempView("raw_texts")

# Use ai_query() for semantic cleaning via SQL (handles VARIANT return)
cleaned_df = spark.sql(f"""
SELECT
  path,
  ai_query(
    '{LLM_ENDPOINT}',
    CONCAT(
      'Clean the following academic paper text. Remove artifacts like broken lines, ',
      'orphaned headers, and page numbers. Preserve the original content, section structure, ',
      'and meaning exactly. Return only the cleaned text, nothing else.\n\n',
      raw_text
    )
  )::STRING AS cleaned_text
FROM raw_texts
""")

cleaned_df.createOrReplaceTempView("cleaned_texts")
display(spark.sql("SELECT path, substring(cleaned_text, 1, 500) AS preview FROM cleaned_texts"))

## E. Chunking Documents

We split the cleaned text into chunks suitable for embedding and vector search. Key parameters:
- **chunk_size**: Maximum tokens per chunk (512 is a common choice)
- **chunk_overlap**: Tokens shared between adjacent chunks (prevents context loss at boundaries)

We use recursive character splitting, which tries to split on paragraph boundaries first, then sentences, then words.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, StringType

# Configure chunking
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Collect to driver for chunking (dataset is small enough)
cleaned_rows = spark.sql("SELECT path, cleaned_text FROM cleaned_texts").collect()

chunks = []
for row in cleaned_rows:
    text_chunks = text_splitter.split_text(row.cleaned_text) if row.cleaned_text else []
    for i, chunk in enumerate(text_chunks):
        chunks.append((row.path, i, chunk))

# Create DataFrame from chunks
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("path", StringType()),
    StructField("chunk_index", IntegerType()),
    StructField("chunk_text", StringType()),
])
chunks_df = spark.createDataFrame(chunks, schema)

# Add unique chunk ID
chunks_flat_df = chunks_df.withColumn(
    "chunk_id",
    F.sha2(F.concat(F.col("path"), F.lit("_"), F.col("chunk_index").cast("string")), 256)
).select("chunk_id", "path", "chunk_index", "chunk_text")

print(f"Total chunks: {chunks_flat_df.count()}")
display(chunks_flat_df)

## F. Save to Delta Table

Save chunks to a Delta table with:
- **Unique ID per row** (`chunk_id`) — required for Vector Search indexing
- **Change Data Feed enabled** — required for Delta Sync Vector Search

In [ ]:
TABLE_NAME = f"{CATALOG}.{SCHEMA}.arxiv_chunks"

# Enable Change Data Feed and write
(
    chunks_flat_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Saved {spark.table(TABLE_NAME).count()} chunks to {TABLE_NAME}")
display(spark.table(TABLE_NAME).limit(5))

## Key Takeaways

1. `ai_parse_document()` extracts text, tables, and images from PDFs — available in both SQL and Python
2. `ai_query()` with a Foundation Model can clean parsed text semantically (higher quality, higher cost)
3. Recursive character splitting preserves semantic boundaries when chunking
4. Each chunk needs a unique ID and the Delta table needs Change Data Feed for Vector Search sync
5. Chunk size and overlap are tunable parameters — smaller chunks = more precise retrieval, larger = more context

## Architecture Diagram

![Architecture Diagram](../../assets/diagrams/lab-01-document-parsing-chunking.png)

## Key Concepts

| Concept | Definition |
|---------|-----------|
| `ai_parse_document()` | Databricks SQL function that extracts text, tables, and images from documents using Mosaic AI |
| `ai_query()` | Databricks SQL function that invokes a Foundation Model endpoint for inference within SQL |
| Recursive Character Splitting | Chunking strategy that tries to split on semantic boundaries (paragraphs > sentences > words) |
| Chunk Overlap | Shared text between adjacent chunks to prevent context loss at boundaries |
| Change Data Feed (CDF) | Delta Lake feature that tracks row-level changes, required for Delta Sync Vector Search |
| Unity Catalog Volume | Managed storage for non-tabular files (PDFs, images, CSVs) within Unity Catalog |

## Exam Preparation

### Common Exam Question Patterns

1. **"Which function parses documents in Databricks?"** → `ai_parse_document()`. Not `ai_query()` (that's for model inference), not `Unstructured` (that's a Python library for local use).

2. **"What are the prerequisites for creating a Delta Sync Vector Search index?"** → Unique primary key column + Change Data Feed enabled on the source Delta table.

3. **"Which cleaning approach has higher quality but higher cost?"** → `ai_query()` with an LLM for semantic cleaning (vs. simple Spark UDF concatenation).

4. **"A vector database can only store 100M embeddings but chunking produces 150M. How to reduce?"** → Increase chunk size → fewer chunks per document → fewer embeddings.

5. **"Why is chunk overlap used?"** → Prevents context from being lost at chunk boundaries. Improves retrieval quality for content spanning adjacent chunks.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Serverless Compute | ~15 min notebook execution | ~$0.50 |
| ai_parse_document | 5-8 PDFs, ~50 pages total | ~$0.25-0.50 |
| ai_query (LLM cleaning) | ~50K tokens input + output | ~$0.25-0.50 |
| Delta Storage | ~5MB chunks table | ~$0.01 |
| **Total** | | **~$1-2** |

**Estimated time:** ~30 min | **Estimated cost:** ~$1-2 (parsing + LLM cleaning tokens)